# variant_enrichment.ipynb
Which pathogenic variants are more common in ecDNA+ samples?  
For each sample, get pathogenic variants and aggregate all pathogenic variants into table, run chi2 test and FDR correction on each gene.  
We ran the test on cancer genes mutated at least 4 times.  
We used subset tumor types with at least 1 gene mutant and threw out tumor types without ecDNA.  

## RESULTS
### Pathogenic variants (n=1479)
- BRAF and CTNNB1 were depleted in amplified tumors.  
- TP53 was enriched in amplified tumors, but depleted in extrachromosomal vs. chromosomally amplified tumors.  
- NOTCH1, EGFR, H3-3A and ATM were enriched in amplified tumors.

Variant table saved to `out/pathogenic_variants.tsv`.  
Enrichment results saved to `out/pathogenic_enrichment_results.tsv`.

### Likely pathogenic variants (n=7071)
- KNL1, PDGFRA, PABPC1, AURKA, STAG2, KNSTRN, TP53, H3-3A enriched in amplified tumors.  
- Not powered to test extrachromosomal vs. chromosomal amplification (min. qvalue 0.3).
- GNAS nominally depleted in ec vs chr (p=0.02).

Variant table saved to `out/likely_pathogenic_variants.tsv`.  
Enrichment results saved to `out/likely_pathogenic_enrichment_results.tsv`.

## Import and format
our data into a DataFrame. This section will generate variant tables of the form:

|biosample_id |	CHROM |	start |	end |	REF |	ALT |	HotSpot |	SYMBOL |	DNA_change |	AA_change |	Consequence |	IMPACT |	PolyPhen |	SIFT |	CLIN_SIG|
|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|
|`str`|`str`|`int`|`int`|`str`|`str`|`dbl`|`str`|`str`|`str`|`str`|`str`|`str`|`str`|`str`|

All values from SYMBOL onwards (inclusive) are comma-separated strings, one per transcript affected.


In [ ]:
import cyvcf2
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
import os

import sys
sys.path.append('../src')
from data_imports import *
from somatic_snv_functions import *

import scipy.stats
from statsmodels.stats.multitest import multipletests

In [ ]:
def parse_CSQ(fields,csq):
    '''
    parse CSQ string into a nice dictionary.
    CSQ fields are separated by '|'; CBTN somatic variant files have 57 or 73 fields; SJC has 66.
    Variants may have more than 1 CSQ associated with multiple transcripts, separated by ','.
    '''
    fields = fields.split('|')
    csqs = csq.split(',')
    csqs = [x.split('|') for x in csqs]
    csqs = list(zip(*csqs))
    #csqs = [set(x) - {''} for x in csqs]
    #return csqs
    assert len(csqs) == len(fields),f'{len(csqs)},{len(fields)}'
    return dict(zip(fields,csqs))

In [ ]:
def parse_vcf(vcf_path):
    '''
    read a vcf file into a pandas dataframe.
    '''
    all_variants = pd.DataFrame()
    vcf = cyvcf2.VCF(vcf_path)
    csq_header = vcf.get_header_type('CSQ')['Description'].split('Format: ')[-1]
    for variant in vcf:
        values ={
            'REF':variant.REF, 
            'ALT':tuple(set(variant.ALT)-{''}), 
            'CHROM':variant.CHROM, 
            'start':variant.start, 
            'end':variant.end, 
            'ID':variant.ID,
            'FILTER':variant.FILTER, 
            'QUAL':variant.QUAL,
            'HotSpot':variant.INFO.get('HotSpotAllele'),
            'CSQ':variant.INFO.get('CSQ')
        }
        csq = parse_CSQ(csq_header,values['CSQ'])
        #print(len(csq))
        values = csq | values
        #print(values)
        values = pd.DataFrame([values])
        keep = ['CHROM','start','end','REF','ALT','HotSpot','SYMBOL','HGVSc','HGVSp','Consequence','IMPACT','PolyPhen','SIFT','CLIN_SIG']
#        keep = ['CHROM','start','end','REF','ALT','HotSpot','SYMBOL','Consequence','IMPACT','CLIN_SIG',
#                "Gene","Feature_type","Feature","HGVSc","HGVSp","Protein_position","Amino_acids","Codons",
#                "VARIANT_CLASS","HGNC_ID","CANONICAL","TSL","CCDS","ENSP","SWISSPROT","TREMBL","UNIPARC",
#               ]
        values = values[keep]
        all_variants = pd.concat([all_variants,values])
    return all_variants

In [ ]:
def get_vcfs(directory):
    for filename in os.listdir(directory):
        full_path = os.path.join(directory, filename)
        if os.path.isfile(full_path) and filename.endswith(".vcf.gz"):
            yield full_path
def extract_sample_name(filepath):
    sample = os.path.basename(filepath)
    sample = sample.split('.')[0]
    return sample

def get_all_variants(directory,dry_run=False,verbose=False):
    '''
    Read all patient genomes,
    filter for pathogenic variants,
    and aggregate into a single pandas dataframe.
    '''
    big_df = pd.DataFrame()
    i=0
    for vcf in get_vcfs(directory):
        i=i+1
        name = extract_sample_name(vcf)
        if verbose:
            print('Now processing sample number',i,':',name)
        variants_df = parse_vcf(vcf)
        variants_df['prefix'] = name
        dfs = [df for df in [big_df,variants_df] if not df.empty]
        big_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
        #big_df = pd.concat([big_df,variants_df])
        if dry_run == True:
            if i > 10:
                break
    # one row per gene affected
    #big_df = big_df.explode('SYMBOL')
    return big_df

def get_cancer_genes(path='cancer_genes.txt'):
    with open(path) as f:
        return {line.strip() for line in f if line.strip()}

def implode(df):
    return df.groupby(['biosample_id','CHROM','start','end','REF','ALT','HotSpot'], as_index=False, dropna=False).agg({
        c: tuple for c in ['SYMBOL','DNA_change','AA_change','Consequence','IMPACT','PolyPhen','SIFT','CLIN_SIG']
    })

def postprocess_variants(df,bs_key,lp=False,cancer_genes=None):
    # use biosample_ids
    df['prefix'] = df['prefix'].map(bs_key)
    df = df.rename(columns={'prefix':'biosample_id'})
    df = df.dropna(axis='index',subset='biosample_id')
    
    # filter variants and annotations
    df = df.explode(['SYMBOL','HGVSc','HGVSp','Consequence','IMPACT','PolyPhen','SIFT','CLIN_SIG'])
    df['DNA_change'] = df.HGVSc.map(lambda x: x.split('.')[-1])
    df['AA_change'] = df.HGVSp.map(lambda x: x.split('.')[-1])
    if lp:
        df = df[
            df.HGVSp.str.startswith('ENSP') &
            df.SYMBOL.isin(cancer_genes) &
            df.IMPACT.isin(['HIGH','MODERATE']) &
            (df.PolyPhen.str.contains('damaging') | (df.PolyPhen == '')) &
            (df.SIFT.str.contains('deleterious') | (df.SIFT == ''))
        ].copy()
    df = df.drop_duplicates(subset=['biosample_id','SYMBOL','DNA_change','AA_change'])
    df = implode(df)
    return df

In [ ]:
biosample_key = read_manifest()
biosamples = import_biosamples()
biosample_key = biosample_key[biosample_key.isin(biosamples.index)]

In [ ]:
p_dir = "./data/filter/pathogenic"
p_df = get_all_variants(p_dir,dry_run=False)
p_df = postprocess_variants(p_df,biosample_key)
print(len(p_df))
p_df.head()

In [ ]:
lp_dir = "./data/filter/likely_pathogenic"
lp_df = get_all_variants(lp_dir,dry_run=False)
lp_df = postprocess_variants(lp_df,biosample_key,lp=True,cancer_genes=get_cancer_genes())
print(len(lp_df))
lp_df.head()

In [ ]:
# write
def write_variant_table(df,outfile):
    tuple_cols = [c for c in df.columns if df[c].map(lambda x:isinstance(x, tuple)).any()]
    for c in tuple_cols:
        df[c] = df[c].apply(
            lambda x: ','.join(map(str, filter(None, x)))
            if isinstance(x, tuple) else x
        )
    df.to_csv(outfile,sep='\t')
write_variant_table(p_df,'out/pathogenic_variants.tsv')
write_variant_table(lp_df,'out/likely_pathogenic_variants.tsv')

In [ ]:
# are all SYMBOL entries in lp_df composed of only 1 unique entry?
# Yes
lp_df.SYMBOL.map(lambda x: len({x})).unique()

In [ ]:
# Which H3 mutations in this dataset?
h3_genes = get_cancer_genes('../data/variants/databases/H3-new-gene-names.txt')
pd.set_option('display.max_rows', 300)
lp_df[lp_df.SYMBOL.map(lambda t: any(x in h3_genes for x in t.split(',')))]

## Tests 
detect more common pathogenic variants in ecDNA+ samples by chi2 test, FDR correction.  

In [ ]:
def format_for_variant_enrichment_tests(p_df,lp_df,biosamples=None):
    # process a df of the format output by postprocess_variants (indexed by sample, variant) and make the following changes
    #    aggregate all variants for each sample
    #    drop tumor types without ecDNA
    #    deduplicate by patient_id
    # returns: dataframe of the form
    #    biosample_id 	sex 	patient_id 	tumor_history 	age_at_diagnosis 	cohort 	extent_of_tumor_resection 	cancer_type 	cancer_subclass 	amplicon_class 	pathogenic_variants 	likely_pathogenic_variants 	in_snv_set 	amplified 	ecDNA

    if biosamples is None:
        biosamples = import_biosamples()
        
    # aggregate
    for s in [(p_df, 'pathogenic'),(lp_df,'likely_pathogenic')]:
        df,label = s
        df = df.copy()
        df = (df
            .assign(variants=df.SYMBOL.str.split(','))
            .explode('variants')
            .groupby('biosample_id')
            .agg({'variants':set})
            .rename(columns=lambda x: f'{label}_{x}')
           )
        biosamples = biosamples.merge(df,how='left',right_index=True,left_index=True)
        biosamples[f'{label}_variants'] = biosamples[f'{label}_variants'].where(biosamples[f'{label}_variants'].notna(), set())
    df = biosamples
    
    # drop tumor types without ecDNA
    ec_tumor_types = df[df.amplicon_class.map(lambda x: 'ecDNA' in x)].cancer_type.unique()
    df = df[df.cancer_type.isin(ec_tumor_types)]
    
    # deduplicate by patient id
    df = df.sort_values(by=['patient_id','amplicon_class','ecDNA_sequences_detected','age_at_diagnosis'],
                   ascending=[True,False,True,True])
    df["in_snv_set"]=~df.duplicated(subset=["patient_id"],keep='last')
    df["amplicon_class"] = df["amplicon_class"].replace({
        "intrachromosomal": "chromosomal",
    })

    # annotate
    df['amplified'] = df.amplicon_class.isin(['ecDNA','chromosomal'])
    df['ecDNA'] = df.amplicon_class == 'ecDNA'
    df = df.drop(columns=['external_sample_id','file_name','ecDNA_sequences_detected','in_unique_tumor_set','in_unique_patient_set'])
    return df

In [ ]:
df = format_for_variant_enrichment_tests(p_df,lp_df,biosamples)
df.head()

In [ ]:
# Which and how many genes are we testing
all_p_variants = set().union(*df['pathogenic_variants'])
whitelist = {'DNMT1','HLA-B','RHEB','TGFBR1',}
test_p_variants = all_p_variants & (get_cancer_genes() | whitelist)

In [ ]:
pd.crosstab(df.ecDNA, df.pathogenic_variants.map(lambda x: 'TP53' in x))

In [ ]:
def test_enrichment(crosstab,min_count=5,verbose=False):
    # test enrichment via chi-sq test on a 2x2 count table.
    t = crosstab
    if t.shape != (2,2):
        return None
    a,b,c,d = t.iloc[1,1], t.iloc[1,0], t.iloc[0,1], t.iloc[0,0]
    if verbose:
        print(gene, a,b,c,d, a+b+c+d)
    if (a+c<min_count) or (a+b<min_count): # need at least 5 mutant samples and 5 amp/ec samples for valid test
        return None
    res = scipy.stats.chi2_contingency([[a,b],[c,d]])
    chi2, p, dof, expected = res
    # code for over/underrepresentation
    if b*c == 0:
        odds_ratio = np.inf
    else:
        odds_ratio = (a*d) / (b*c)
    enriched = True if odds_ratio >1 else False
    return [chi2,odds_ratio,p]

def test_all_gene_enrichments(df,column,gene_whitelist=None,cancer_type='all'):
    # test all variants for enrichment in amplified or extrachromosomally amplified samples.
    # col: either 'pathogenic_variants' or 'likely_pathogenic_variants'
    if cancer_type != 'all':
        df = df[df.cancer_type == cancer_type]
    genes = set().union(*df[column])
    if gene_whitelist is not None:
        genes = genes & (gene_whitelist)
    results = []
    not_tested = [pd.NA, pd.NA, pd.NA]
    for gene in genes:
        # test amplification
        tbl = pd.crosstab(df.amplified,df[column].map(lambda x: gene in x))
        res1 = test_enrichment(tbl)
        # test extrachromosomal
        if res1 is not None:
            subdf = df[df.amplified]
            tbl = pd.crosstab(subdf.ecDNA,subdf[column].map(lambda x: gene in x))
            res2 = test_enrichment(tbl)
            if res2 is None:
                res2 = not_tested
#        else:
#            res1 = not_tested
#            res2 = not_tested
            results.append([gene,cancer_type]+res1+res2)
    result_cols = pd.MultiIndex.from_arrays([
        ['gene','cancer_type']+['amplified']*3+['ecDNA']*3,
        ['']*2+['chi2','odds_ratio','p-value']*2
    ])
    result_df = pd.DataFrame(results,columns=result_cols)
    return result_df

def correct_gene_enrichment_fdr(df):
    # multiple hypothesis (FDR) correction for output of test_all_gene_enrichments
    df = df.copy()
    _,q,_,_ = multipletests(df[('amplified','p-value')],method='fdr_bh')
    df.insert(
        loc=5,
        column=('amplified','q-value'),
        value=q
    )
    _,q,_,_ = multipletests(df[('ecDNA','p-value')].dropna(),method='fdr_bh')
    df[('ecDNA','q-value')]=pd.NA
    mask=df[('ecDNA','p-value')].notna()
    df.loc[mask,('ecDNA','q-value')]=q
    return df

def filter_gene_enrichment_results(df,col='q-value',alpha=0.1):
    # filter a gene enrichment result table for interesting hits
    return df[(df[('amplified',col)] < alpha) | (df[('ecDNA',col)] < alpha)]

In [ ]:
lp_test_tbl = test_all_gene_enrichments(df,'likely_pathogenic_variants')
lp_test_tbl = correct_gene_enrichment_fdr(lp_test_tbl)
print(min(lp_test_tbl[('ecDNA','q-value')].dropna()))
lp_test_tbl.to_csv('out/likely_pathogenic_enrichment_results.tsv')
filter_gene_enrichment_results(lp_test_tbl)

In [ ]:
filter_gene_enrichment_results(lp_test_tbl,col='p-value',alpha=0.5)

In [ ]:
p_test_tbl = test_all_gene_enrichments(df,'pathogenic_variants',test_p_variants)
p_test_tbl = correct_gene_enrichment_fdr(p_test_tbl)
print(min(p_test_tbl[('ecDNA','q-value')].dropna()))
p_test_tbl.to_csv('out/pathogenic_enrichment_results.tsv',sep='\t')
filter_gene_enrichment_results(p_test_tbl)

In [ ]:
## Hits
pd.crosstab(df.amplicon_class, df.pathogenic_variants.map(lambda x: 'TP53' in x))